# Notebook 01 — Construction de la matrice de features (logique Kaggle)

## Contexte
Le projet s’appuie sur le dataset *Home Credit Default Risk*. Les données sont réparties en plusieurs tables (application, bureau, previous_application, etc.).
La stratégie utilisée ici est celle du kernel Kaggle classique :  
- **1 table → 1 fonction de preprocessing / agrégation**
- **jointure progressive** sur la clé `SK_ID_CURR`
- sortie finale : une **matrice de features tabulaire** prête pour l’entraînement

## Objectif du notebook
- Exécuter le pipeline de feature engineering **reproductible** implémenté dans `src/credexp/data/build_features.py`
- Générer et sauvegarder `data/processed/features.parquet`
- Vérifier rapidement la forme et la séparation train/test via `TARGET`

> Remarque : le code “lourd” est dans un module Python (et non dans le notebook) pour garantir la reproductibilité et éviter les incohérences d’état.


In [1]:
from pathlib import Path

import pandas as pd

from credexp.config import DATA_DIR
from credexp.data.build_features import build_feature_matrix
from credexp.data.io import load_parquet, save_parquet


## Pré-requis : données brutes

Les fichiers Kaggle doivent être présents dans `data/raw/` (noms exacts) :

- application_train.csv
- application_test.csv
- bureau.csv
- bureau_balance.csv
- previous_application.csv
- POS_CASH_balance.csv
- installments_payments.csv
- credit_card_balance.csv


In [2]:
raw_dir = DATA_DIR / "raw"
required = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "POS_CASH_balance.csv",
    "installments_payments.csv",
    "credit_card_balance.csv",
]

missing = [f for f in required if not (raw_dir / f).exists()]
missing


[]

Si la liste `missing` est vide, on peut construire la matrice de features.
Pour une exécution rapide de validation, on peut utiliser un mode "debug" (subset).
Ici, on exécute la version complète (comme pour l’entraînement réel).


In [3]:
out_path = DATA_DIR / "processed" / "features.parquet"

df = build_feature_matrix(raw_path=raw_dir, debug=False)
save_parquet(df, out_path)

df.shape


{"ts": "2026-02-18T19:51:23Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\raw\\application_train.csv"}
{"ts": "2026-02-18T19:51:25Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\raw\\application_test.csv"}
{"ts": "2026-02-18T19:51:25Z", "level": "INFO", "name": "credexp.data.build_features", "msg": "Train samples: 307511, test samples: 48744"}
{"ts": "2026-02-18T19:51:26Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\raw\\bureau.csv"}
{"ts": "2026-02-18T19:51:28Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\raw\\bureau_balance.csv"}
{"ts": "2026-02-18T19:51:38Z", "level": "INFO", "name": "credexp.data.build_features", "msg": "Bureau df shape: (305811, 116)"}
{"ts": "2026-02-18T19:51:38Z", "level": "INFO", "name": "credexp.da

(356251, 797)

In [4]:
df2 = load_parquet(out_path)
df2.shape, df2.columns[:10]


{"ts": "2026-02-18T19:52:38Z", "level": "INFO", "name": "credexp.data.io", "msg": "load_parquet path=G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\processed\\features.parquet"}


((356251, 797),
 Index(['SK_ID_CURR', 'TARGET', 'CODE_GENDER', 'FLAG_OWN_CAR',
        'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
        'AMT_ANNUITY', 'AMT_GOODS_PRICE'],
       dtype='object'))

## Vérification structure : séparation train/test

Le dataset final contient :
- des lignes d’entraînement : `TARGET` non-null
- des lignes test Kaggle : `TARGET` null

On vérifie que cette séparation est cohérente.


In [5]:
assert "TARGET" in df2.columns

n_train = int(df2["TARGET"].notna().sum())
n_test = int(df2["TARGET"].isna().sum())
n_train, n_test


(307507, 48744)

## Conclusion

Nous avons généré une matrice de features unique (`features.parquet`) de manière reproductible, en suivant la logique du kernel Kaggle :
- agrégations par table
- jointures sur `SK_ID_CURR`
- séparation train/test via `TARGET`

Cette matrice servira de base **unique** pour :
- l’EDA (Notebook 02)
- l’entraînement + suivi MLflow (Notebook 03)
- la partie déploiement (Partie 2) avec un pipeline cohérent.
